## Myopic Adaptive 

In [1]:
using CSV, DataFrames, Gurobi

# --------------------------------------------------------
# 1. Load & Clean
# --------------------------------------------------------

df = CSV.read("output.csv", DataFrame)

# Clean all string columns (safe)
for col in names(df)
    if eltype(df[!, col]) <: AbstractString
        df[!, col] = String.(df[!, col])
        df[!, col] = allowmissing(df[!, col])
        replace!(df[!, col], "-" => missing)
        replace!(df[!, col], "–" => missing)
        replace!(df[!, col], ""  => missing)
    end
end

# Remove rows missing important model fields
df = filter(row -> !ismissing(row.FPTS) &&
                   !ismissing(row.POSITION) &&
                   !ismissing(row.ADP_AVG), df)

# Standard cleanup
df.POSITION = strip.(df.POSITION)
df.player_id = 1:nrow(df)

# --------------------------------------------------------
# 2. Create lookup dictionaries (KEEP df AS A DATAFRAME)
# --------------------------------------------------------

players  = collect(df.player_id)                    # player IDs
FPTS     = Dict(df.player_id .=> df.FPTS)           # id -> projection
pos      = Dict(df.player_id .=> df.POSITION)       # id -> position
adp      = Dict(df.player_id .=> df.ADP_AVG)        # id -> ADP
bye_week = Dict(df.player_id .=> df.BYE_WEEK)       # id -> bye week

# ⭐ NEW:
upside   = Dict(df.player_id .=> df.UPSIDE)         # id -> upside score
bust     = Dict(df.player_id .=> df.BUST)           # id -> bust score


# --------------------------------------------------------
# 3. Draft config
# --------------------------------------------------------

num_teams = 12
rounds    = 16
my_slot   = 3

T_max = num_teams * rounds
picks = 1:T_max

function get_snake_picks(my_slot::Int, num_teams::Int, roster_size::Int)
    picks = Int[]
    for r in 1:roster_size
        if isodd(r)
            push!(picks, (r-1)*num_teams + my_slot)
        else
            push!(picks, r*num_teams + 1 - my_slot)
        end
    end
    return picks
end

my_picks = get_snake_picks(my_slot, num_teams, rounds)

# --------------------------------------------------------
# 4. Roster constraints
# --------------------------------------------------------

min_slots = Dict(
    "QB"   => 1,
    "RB"   => 2,
    "WR"   => 2,
    "TE"   => 1,
    "FLEX" => 1,
    "DST"  => 1,
    "K"    => 1
)

flex_positions = Set(["RB","WR","TE"])

max_slots = Dict(
    "QB"  => 2,
    "RB"  => 5,
    "WR"  => 5,
    "TE"  => 2,
    "K"   => 1,
    "DST" => 2
)

current_count = Dict(
    "QB"  => 0,
    "RB"  => 0,
    "WR"  => 0,
    "TE"  => 0,
    "K"   => 0,
    "DST" => 0
)

# --------------------------------------------------------
# 5. Feasibility function
# --------------------------------------------------------

function feasible_pick(p, current_count, min_slots, rounds_left)
    c = deepcopy(current_count)
    c[p] += 1

    remaining_required = 0

    for posname in keys(min_slots)
        if posname == "FLEX"
            needed = min_slots["FLEX"]
            have   = sum(c[x] for x in ["RB", "WR", "TE"])
            if have < needed
                remaining_required += (needed - have)
            end
        else
            if c[posname] < min_slots[posname]
                remaining_required += (min_slots[posname] - c[posname])
            end
        end
    end

    return remaining_required <= rounds_left
end

# --------------------------------------------------------
# 6. Myopic Auto-Drafter 1 vs ADP opponents
# --------------------------------------------------------

sorted_players = sort(players, by = i -> adp[i])
taken = Set{Int}()

my_team = DataFrame(
    pick=Int[], player_id=Int[], ADP=Float64[], PLAYER=String[],
    POSITION=String[], FPTS=Float64[], BYE_WEEK=Float64[]
)

for t in picks
    if t in my_picks

        available = [i for i in players if !(i in taken)]
        picks_left = length(my_picks[my_picks .>= t])

        eligible = [
            i for i in available
            if current_count[pos[i]] < max_slots[pos[i]] &&
               feasible_pick(pos[i], current_count, min_slots, picks_left - 1)
        ]

        if isempty(eligible)
            eligible = available
        end

        selected = argmax(i -> FPTS[i], eligible)

        push!(my_team, (
            t,
            selected,
            adp[selected],
            df.PLAYER[selected],
            pos[selected],
            FPTS[selected],
            bye_week[selected]
        ))

        current_count[pos[selected]] += 1
        push!(taken, selected)

    else
        # Opponents pick best available by ADP
        for i in sorted_players
            if !(i in taken)
                push!(taken, i)
                break
            end
        end
    end
end

println("Myopic baseline team:")
println(my_team)
println("Total projected FPTS = ", sum(my_team.FPTS))
println("Position counts:", current_count)

Myopic baseline team:
16×7 DataFrame
 Row │ pick   player_id  ADP      PLAYER               POSITION  FPTS     BYE_WEEK
     │ Int64  Int64      Float64  String               String    Float64  Float64
─────┼─────────────────────────────────────────────────────────────────────────────
   1 │     3         25     20.2  Josh Allen           QB          376.6       7.0
   2 │    22         27     21.5  Lamar Jackson        QB          372.4       7.0
   3 │    27         36     32.5  Tyreek Hill          WR          259.9      12.0
   4 │    46         42     49.3  DK Metcalf           WR          234.5       5.0
   5 │    51         51     54.8  Xavier Worthy        WR          228.5      10.0
   6 │    70         81     74.5  Jerry Jeudy          WR          219.7       9.0
   7 │    75        103     90.8  Jakobi Meyers        WR          197.9       8.0
   8 │    94        183    107.2  Brandon Aubrey       K           140.1      10.0
   9 │    99        112    112.8  Austin Ekeler   

# ESPN Baseline

In [2]:
println("\n================ ESPN-STYLE BASELINE (Baseline 0) ================")

# Reset roster counts
current_count = Dict(
    "QB"  => 0,
    "RB"  => 0,
    "WR"  => 0,
    "TE"  => 0,
    "K"   => 0,
    "DST" => 0,
)

# ESPN fallback: if ESPN is missing, use ADP_AVG instead
df.unified_ESPN = [ismissing(df.ESPN[i]) ? df.ADP_AVG[i] : df.ESPN[i] for i in 1:nrow(df)]


# 👉 ESPN-specific ADP ranking
# Replace :ESPN_ADP with the correct column name from your CSV
ESPN_rank = Dict(df.player_id .=> df.unified_ESPN)

sorted_players = sort(players, by = i -> ESPN_rank[i])

taken = Set{Int}()

espn_team = DataFrame(
    pick      = Int[],
    player_id = Int[],
    PLAYER    = String[],
    POSITION  = String[],
    FPTS      = Float64[],
    unified_ESPN = Float64[],
    BYE_WEEK  = Float64[]
)

for t in picks

    if t in my_picks
        available   = [i for i in players if !(i in taken)]
        picks_left  = length(my_picks[my_picks .>= t])

        selected = nothing

        # ESPN logic → walk ESPN's ADP list top-down
        for i in sorted_players
            if i in taken
                continue
            end

            p = pos[i]

            if current_count[p] < max_slots[p] &&
               feasible_pick(p, current_count, min_slots, picks_left - 1)

                selected = i
                break
            end
        end

        if selected === nothing
            for i in sorted_players
                if !(i in taken)
                    selected = i
                    break
                end
            end
        end

        push!(espn_team, (
            t,
            selected,
            df.PLAYER[selected],
            pos[selected],
            FPTS[selected],
            ESPN_rank[selected],
            bye_week[selected]
        ))

        current_count[pos[selected]] += 1
        push!(taken, selected)

    else
        for i in sorted_players
            if !(i in taken)
                push!(taken, i)
                break
            end
        end
    end
end

println("\nESPN Autodraft Team:")
println(espn_team)
println("\nTotal projected FPTS = ", sum(espn_team.FPTS))
println("Position counts: ", current_count)


================ ESPN-STYLE BASELINE (Baseline 0) ================

ESPN Autodraft Team:
16×7 DataFrame
 Row │ pick   player_id  PLAYER            POSITION  FPTS     unified_ESPN  BYE_WEEK
     │ Int64  Int64      String            String    Float64  Float64       Float64
─────┼───────────────────────────────────────────────────────────────────────────────
   1 │     3          3  Saquon Barkley    RB          278.2           3.0       9.0
   2 │    22         21  Bucky Irving      RB          209.9          22.0       9.0
   3 │    27         32  Jayden Daniels    QB          354.7          27.0      12.0
   4 │    46         29  Mike Evans        WR          246.0          46.0       9.0
   5 │    51         47  Breece Hall       RB          178.7          51.0       9.0
   6 │    70         66  Aaron Jones Sr.   RB          158.6          70.0       6.0
   7 │    75         72  Rome Odunze       WR          193.2          75.0       5.0
   8 │    94         84  Deebo Samuel Sr.  WR

# Greedy Baseline

In [3]:
using JuMP, Gurobi

println("\n================ GREEDY WITH REACHING PENALTY (Baseline 2) ================")

# --------------------------------------------------------
# Pick value calculation (Tuned Exponential Model)
# --------------------------------------------------------

function calculate_pick_value(pick_num::Int)
    return 400.0 * exp(-pick_num / 80.0)
end

# Build pick values dictionary
PICK_VALUES = Dict{Int, Float64}()
for pick in 1:192
    PICK_VALUES[pick] = calculate_pick_value(pick)
end

println("✅ Using Tuned Exponential Model for Pick Values")

# --------------------------------------------------------
# Helper functions for penalty calculation
# --------------------------------------------------------

function acceptable_reach(pick_num::Int)
    round_num = ceil(Int, pick_num / 12)
    
    if round_num == 1
        return 2.0
    elseif round_num == 2
        return 3.0
    elseif round_num == 3
        return 5.0
    elseif round_num == 4
        return 6.0
    elseif round_num <= 8
        return round_num + 4.0  # Rounds 5-8: 9, 10, 11, 12
    else
        return 15.0 + (round_num - 9) * 2  # Round 9: 15, Round 10: 17, etc.
    end
end

function penalty_weight(pick_num::Int)
    round_num = ceil(Int, pick_num / 12)
    
    if round_num <= 3
        return 1.0      # Severe penalty rounds 1-3
    elseif round_num <= 6
        return 0.5      # Moderate penalty rounds 4-6
    elseif round_num <= 8
        return 0.2      # Light penalty rounds 7-8
    else
        return 0.02     # Nearly no penalty rounds 9+
    end
end

function calculate_penalty(player_id::Int, pick_num::Int)
    player_adp = adp[player_id]
    
    # How many picks early are we drafting them?
    picks_early = player_adp - pick_num
    
    # If picks_early <= 0, we're drafting at or after their ADP (value!), no penalty
    if picks_early <= 0
        return 0.0
    end
    
    # We're drafting early - check if it's within acceptable range
    acceptable = acceptable_reach(pick_num)
    
    if picks_early <= acceptable
        # Within acceptable reach, no penalty
        return 0.0
    else
        # Exceeds acceptable reach, apply penalty
        excess_reach = picks_early - acceptable
        λ = penalty_weight(pick_num)
        pick_val = get(PICK_VALUES, pick_num, 100.0)
        return λ * excess_reach * pick_val
    end
end

# --------------------------------------------------------
# Reset for new baseline
# --------------------------------------------------------

current_count = Dict(
    "QB"  => 0,
    "RB"  => 0,
    "WR"  => 0,
    "TE"  => 0,
    "K"   => 0,
    "DST" => 0,
)

sorted_players = sort(players, by = i -> adp[i])
taken = Set{Int}()

greedy_team = DataFrame(
    pick      = Int[],
    player_id = Int[],
    PLAYER    = String[],
    POSITION  = String[],
    FPTS      = Float64[],
    ADP       = Float64[],
    penalty   = Float64[],
    net_value = Float64[],
    BYE_WEEK=Float64[]
)

# --------------------------------------------------------
# Main draft loop
# --------------------------------------------------------

for t in picks
    if t in my_picks
        println("\n--- Your pick: $t (Round $(ceil(Int, t/12))) ---")
        
        available = [i for i in players if !(i in taken)]
        remaining_picks = my_picks[my_picks .>= t]
        picks_left = length(remaining_picks)
        
        # Build optimization model for remaining picks
        model = Model(Gurobi.Optimizer)
        set_silent(model)
        
        # Decision variables: x[i,p] = 1 if player i selected at pick p
        @variable(model, x[i=available, p=remaining_picks], Bin)
        
        # Objective: maximize (FPTS - penalty)
        @objective(model, Max,
            sum((FPTS[i] - calculate_penalty(i, p)) * x[i,p] 
                for i in available, p in remaining_picks)
        )
        
        # Constraint 1: Pick exactly one player at each of my remaining picks
        for p in remaining_picks
            @constraint(model, sum(x[i,p] for i in available) == 1)
        end
        
        # Constraint 2: Each player drafted at most once
        for i in available
            @constraint(model, sum(x[i,p] for p in remaining_picks) <= 1)
        end
        
        # Constraint 3: Position minimums (considering what we already have)
        for (position, min_needed) in min_slots
            if position == "FLEX"
                # FLEX means total RB+WR+TE >= 2+2+1+1 = 6
                flex_already = sum(current_count[pos_name] for pos_name in ["RB", "WR", "TE"])
                flex_needed = 2 + 2 + 1 + 1  # RB_min + WR_min + TE_min + FLEX_min
                
                # Filter flex-eligible players BEFORE the sum
                flex_available = [i for i in available if pos[i] in ["RB", "WR", "TE"]]
                
                @constraint(model,
                    sum(x[i,p] for i in flex_available, p in remaining_picks) 
                    >= flex_needed - flex_already
                )
            else
                already_have = current_count[position]
                still_need = max(0, min_needed - already_have)
                
                # Filter position-specific players BEFORE the sum
                pos_available = [i for i in available if pos[i] == position]
                
                @constraint(model,
                    sum(x[i,p] for i in pos_available, p in remaining_picks) 
                    >= still_need
                )
            end
        end

        # Constraint 4: Position maximums
        for (position, max_allowed) in max_slots
            already_have = current_count[position]
            can_still_draft = max_allowed - already_have
            
            # Filter position-specific players BEFORE the sum
            pos_available = [i for i in available if pos[i] == position]
            
            @constraint(model,
                sum(x[i,p] for i in pos_available, p in remaining_picks) 
                <= can_still_draft
            )
        end
        
        # Solve the optimization problem
        optimize!(model)
        
        if termination_status(model) != MOI.OPTIMAL
            println("Warning: No optimal solution found at pick $t")
            # Fallback to myopic if optimization fails
            eligible = [
                i for i in available
                if current_count[pos[i]] < max_slots[pos[i]] &&
                   feasible_pick(pos[i], current_count, min_slots, picks_left - 1)
            ]
            selected = argmax(i -> FPTS[i], eligible)
        else
            # Extract the player selected for current pick t
            selected = nothing
            for i in available
                if value(x[i,t]) > 0.5
                    selected = i
                    break
                end
            end
            
            if selected === nothing
                println("Warning: No player selected at pick $t, using fallback")
                eligible = [
                    i for i in available
                    if current_count[pos[i]] < max_slots[pos[i]] &&
                       feasible_pick(pos[i], current_count, min_slots, picks_left - 1)
                ]
                selected = argmax(i -> FPTS[i], eligible)
            end
        end
        
        # Calculate penalty for selected player
        pen = calculate_penalty(selected, t)
        net_val = FPTS[selected] - pen
        
        println("Selected: $(df.PLAYER[selected]) ($(pos[selected])) - FPTS: $(FPTS[selected]), ADP: $(adp[selected]), Penalty: $(round(pen, digits=2)), Net: $(round(net_val, digits=2))")
        
        push!(greedy_team, (
            t,
            selected,
            df.PLAYER[selected],
            pos[selected],
            FPTS[selected],
            adp[selected],
            pen,
            net_val,
            bye_week[selected]
        ))
        
        current_count[pos[selected]] += 1
        push!(taken, selected)
        
    else
        # Opponents pick best available by ADP
        for i in sorted_players
            if !(i in taken)
                push!(taken, i)
                break
            end
        end
    end
end

println("\n\n================ GREEDY TEAM RESULTS ================")
println(greedy_team)
println("\nTotal projected FPTS = ", sum(greedy_team.FPTS))
println("Total penalty = ", sum(greedy_team.penalty))
println("Net value = ", sum(greedy_team.net_value))
println("\nPosition counts: ", current_count)


================ GREEDY WITH REACHING PENALTY (Baseline 2) ================
✅ Using Tuned Exponential Model for Pick Values

--- Your pick: 3 (Round 1) ---
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Selected: Jahmyr Gibbs (RB) - FPTS: 259.5, ADP: 4.2, Penalty: 0.0, Net: 259.5

--- Your pick: 22 (Round 2) ---
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Selected: Kyren Williams (RB) - FPTS: 221.3, ADP: 24.2, Penalty: 0.0, Net: 221.3

--- Your pick: 27 (Round 3) ---
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Selected: Trey McBride (TE) - FPTS: 139.3, ADP: 28.0, Penalty: 0.0, Net: 139.3

--- Your pick: 46 (Round 4) ---
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Selected: Patrick Mahomes II (QB) - FPTS: 328.1, ADP: 49.7, Penalty: 0.0, Net: 328.1

--- Your pick: 51 (Round 5) ---
Set parameter Username
A

## Greedy Drafter with Reaching Penalty - How It Works

### Core Concept
The model optimizes draft picks by maximizing projected fantasy points while penalizing "reaches" - drafting players significantly earlier than their Average Draft Position (ADP).

### Penalty Formula
```
Penalty = λ × Excess Reach × Pick Value

where:
- Excess Reach = max(0, (ADP - Current Pick) - Acceptable Reach)
- λ = Penalty weight (varies by round)
- Pick Value = Opportunity cost of the pick (exponential decay)
```

### Key Components

**1. Acceptable Reach (by round)**
- Round 1: 2 picks
- Round 2: 3 picks  
- Round 3: 5 picks
- Round 4: 6 picks
- Rounds 5-8: 9-12 picks (gradually increasing)
- Rounds 9+: 15+ picks (very forgiving for sleepers)

**2. Penalty Weight (λ)**
- Rounds 1-3: λ = 1.0 (severe)
- Rounds 4-6: λ = 0.5 (moderate)
- Rounds 7-8: λ = 0.2 (light)
- Rounds 9+: λ = 0.02 (minimal)

**3. Pick Value (Tuned Exponential)**
```
Pick Value = 400 × e^(-pick/80)
```
- Pick 7: ~366 points
- Pick 55: ~201 points
- Pick 127: ~82 points

Early picks have high opportunity cost; late picks are cheap to "waste."

### Examples

**Example 1: Big Reach Early (Bad)**
- Draft QB at pick 7 with ADP 62.5
- Excess reach: 62.5 - 7 - 2 = 53.5 picks
- Penalty: 1.0 × 53.5 × 366 = **19,581 points**
- Result: Heavily discouraged

**Example 2: Small Reach (OK)**  
- Draft RB at pick 31 with ADP 35.8
- Excess reach: 35.8 - 31 - 5 = 0 picks
- Penalty: **0 points**
- Result: Within acceptable range

**Example 3: Late Round Sleeper (Fine)**
- Draft player at pick 127 with ADP 180
- Excess reach: 180 - 127 - 19 = 34 picks
- Penalty: 0.02 × 34 × 82 = **56 points**
- Result: Small penalty, sleeper hunting encouraged

# Optimization 1: Static Optimization

In [4]:
###############################################################
#        GLOBAL OPTIMIZATION MODEL (CONSISTENT WITH ADP BOTS) #
###############################################################

using JuMP, Gurobi, DataFrames

# ============================================================
# 1. Generate Your Pick Numbers Dynamically (snake draft)
# ============================================================

function generate_my_picks(draft_slot::Int; teams::Int=12, rounds::Int=16)
    picks = Int[]
    for r in 1:rounds
        if isodd(r)
            # Odd rounds: order 1 → teams
            push!(picks, (r - 1) * teams + draft_slot)
        else
            # Even rounds: order teams → 1 (snake)
            push!(picks, r * teams - draft_slot + 1)
        end
    end
    return picks
end


# ============================================================
# 2. Pre-compute ADP ranks (1 = earliest, N = latest)
#    This matches the world where everyone else drafts by ADP.
# ============================================================

# players, adp, FPTS, pos, bye_week, min_slots, max_slots, df
# are assumed to be already defined earlier in the notebook.

player_order = sort(players, by = i -> adp[i])   # ascending ADP
adp_rank = Dict{Int,Int}()                       # player_id -> ADP rank

for (k, i) in enumerate(player_order)
    adp_rank[i] = k
end

# If you want a tiny bit of optimism (allow a guy to fall a few spots past rank),
# you can set rank_slack > 0 below. rank_slack = 0 is exactly consistent with ADP bots.


# ============================================================
# 3. Global Optimization Model (Model 1)
# ============================================================

function solve_global_opt_model(
        players,
        my_picks;
        FPTS       = FPTS,
        pos        = pos,
        adp        = adp,
        bye_week   = bye_week,
        min_slots  = min_slots,
        max_slots  = max_slots,
        rank_slack = 0  # how many picks past ADP rank a player might "fall"
    )

    slots = collect(1:length(my_picks))                     # your roster slots 1..R
    slot_to_pick = Dict(s => my_picks[s] for s in slots)    # slot -> overall pick number

    model = Model(Gurobi.Optimizer)
    set_silent(model)

    # Decision variables:
    # x[i,s] = 1 if player i is drafted by us in slot s
    @variable(model, x[players, slots], Bin)

    # y[i] = 1 if player i is on our final roster at any slot
    @variable(model, y[players], Bin)


    # --------------------------------------------------------
    # A. Slot and Uniqueness Constraints
    # --------------------------------------------------------

    # One player per slot
    for s in slots
        @constraint(model, sum(x[i,s] for i in players) == 1)
    end

    # Each player at most once
    for i in players
        @constraint(model, sum(x[i,s] for s in slots) <= 1)
    end

    # Link x and y: if a player appears in any slot, y[i] = 1
    for i in players
        @constraint(model, y[i] >= (1/length(slots)) * sum(x[i,s] for s in slots))
        @constraint(model, y[i] <= sum(x[i,s] for s in slots))
    end


    # --------------------------------------------------------
    # B. ADP-Based Availability (Principled, ADP-Bot-Consistent)
    # --------------------------------------------------------
    # In a pure ADP-bot world:
    #   - A player with ADP rank r_i cannot still be on the board
    #     after pick t > r_i.
    #
    # We implement:
    #   if pick_num > adp_rank[i] + rank_slack  =>  x[i,s] = 0

    for i in players, s in slots
        pick_num = slot_to_pick[s]
        r_i = adp_rank[i]

        if pick_num > r_i + rank_slack
            @constraint(model, x[i,s] == 0)
        end
    end


    # --------------------------------------------------------
    # C. Position Feasibility Constraints (Baseline 2 logic)
    # --------------------------------------------------------

    # Minimums (including FLEX)
    for (position, min_needed) in min_slots
        if position == "FLEX"
            # FLEX = any RB/WR/TE
            @constraint(model,
                sum(y[i] for i in players if pos[i] in ["RB", "WR", "TE"]) >= min_needed
            )
        else
            @constraint(model,
                sum(y[i] for i in players if pos[i] == position) >= min_needed
            )
        end
    end

    # Maximums (skip FLEX max if present)
    for (position, max_allowed) in max_slots
        if position == "FLEX"
            continue
        end
        @constraint(model,
            sum(y[i] for i in players if pos[i] == position) <= max_allowed
        )
    end

    # Total roster size equals number of our picks (e.g., 16)
    @constraint(model, sum(y[i] for i in players) == length(slots))


    # --------------------------------------------------------
    # D. Objective: Maximize Fantasy Points – Reach Penalty
    # --------------------------------------------------------
    @objective(model, Max,
        sum(
            (FPTS[i] - calculate_penalty(i, slot_to_pick[s])) * x[i,s]
            for i in players, s in slots
        )
    )

    optimize!(model)


    # --------------------------------------------------------
    # E. Extract Optimized Team
    # --------------------------------------------------------

    opt_1_team = DataFrame(
        pick      = Int[],
        player_id = Int[],
        PLAYER    = String[],
        POSITION  = String[],
        FPTS      = Float64[],
        ADP       = Float64[],
        penalty   = Float64[],
        net_value = Float64[],
        BYE_WEEK  = Float64[]
    )

    for s in slots
        pick_num = slot_to_pick[s]

        chosen = nothing
        for i in players
            if value(x[i,s]) > 0.5
                chosen = i
                break
            end
        end

        if chosen === nothing
            # This should not happen if the model is feasible & optimal,
            # but we add a fallback for safety.
            remaining = setdiff(players, opt_1_team.player_id)
            chosen = argmax(i -> FPTS[i], remaining)
        end

        pen = calculate_penalty(chosen, pick_num)
        net_val = FPTS[chosen] - pen

        push!(opt_1_team, (
            pick_num,
            chosen,
            df.PLAYER[chosen],
            pos[chosen],
            FPTS[chosen],
            adp[chosen],
            pen,
            net_val,
            bye_week[chosen]
        ))
    end

    sort!(opt_1_team, :pick)
    return opt_1_team
end


###############################################################
# 4. Example Usage – Choose Any Draft Slot
###############################################################

draft_slot = 3 # <--- change this freely from 1 to 12

my_picks = generate_my_picks(draft_slot)
println("\nYour picks from slot $draft_slot: ", my_picks)

global_opt_team = solve_global_opt_model(players, my_picks; rank_slack=0)

println("\n================ GLOBAL OPTIMIZED TEAM (Slot $draft_slot) ================")
println(global_opt_team[:, [:pick, :PLAYER, :POSITION, :ADP, :FPTS, :penalty, :net_value]])

println("\nTotal FPTS    = ", sum(global_opt_team.FPTS))
println("Total Penalty = ", sum(global_opt_team.penalty))
println("Net Value     = ", sum(global_opt_team.net_value))
###############################################################


Your picks from slot 3: [3, 22, 27, 46, 51, 70, 75, 94, 99, 118, 123, 142, 147, 166, 171, 190]
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20

================ GLOBAL OPTIMIZED TEAM (Slot 3) ================
16×7 DataFrame
 Row │ pick   PLAYER                  POSITION  ADP      FPTS     penalty  net_value
     │ Int64  String                  String    Float64  Float64  Float64  Float64
─────┼───────────────────────────────────────────────────────────────────────────────
   1 │     3  Saquon Barkley          RB            3.7    278.2  0.0        278.2
   2 │    22  Kyren Williams          RB           24.2    221.3  0.0        221.3
   3 │    27  James Cook III          RB           29.5    202.1  0.0        202.1
   4 │    46  James Conner            RB           47.3    189.9  0.0        189.9
   5 │    51  Xavier Worthy           WR           54.8    228.5  0.0        228.5
   6 │    70  Jerry Jeudy             WR           74.5    219.

# Optimization 2: Bye Week Constraint Optimization

In [5]:
function solve_global_opt_model2(
        players,
        my_picks;
        FPTS       = FPTS,
        pos        = pos,
        adp        = adp,
        bye_week   = bye_week,
        upside     = upside,   
        bust       = bust,
        min_slots  = min_slots,
        max_slots  = max_slots,
        rank_slack = 0  # how many picks past ADP rank a player might "fall"
    )

    slots = collect(1:length(my_picks))                     # your roster slots 1..R
    slot_to_pick = Dict(s => my_picks[s] for s in slots)    # slot -> overall pick number

    model = Model(Gurobi.Optimizer)
    set_silent(model)

    # Decision variables:
    # x[i,s] = 1 if player i is drafted by us in slot s
    @variable(model, x[players, slots], Bin)

    # y[i] = 1 if player i is on our final roster at any slot
    @variable(model, y[players], Bin)


    # --------------------------------------------------------
    # A. Slot and Uniqueness Constraints
    # --------------------------------------------------------

    # One player per slot
    for s in slots
        @constraint(model, sum(x[i,s] for i in players) == 1)
    end

    # Each player at most once
    for i in players
        @constraint(model, sum(x[i,s] for s in slots) <= 1)
    end

    # Link x and y: if a player appears in any slot, y[i] = 1
    for i in players
        @constraint(model, y[i] >= (1/length(slots)) * sum(x[i,s] for s in slots))
        @constraint(model, y[i] <= sum(x[i,s] for s in slots))
    end
    # --------------------------------------------------------
    # QB Constraint: At most 1 QB in your first 4 picks
    # --------------------------------------------------------

    top4_slots = slots[1:5]

    @constraint(model,
        sum(x[i,s] for i in players for s in top4_slots if pos[i] == "QB") <= 1
    )


    # --------------------------------------------------------
    # B. ADP-Based Availability (Principled, ADP-Bot-Consistent)
    # --------------------------------------------------------

    for i in players, s in slots
        pick_num = slot_to_pick[s]
        r_i = adp_rank[i]

        if pick_num > r_i + rank_slack
            @constraint(model, x[i,s] == 0)
        end
    end


    # --------------------------------------------------------
    # C. Position Feasibility Constraints (Baseline 2 logic)
    # --------------------------------------------------------

    # Minimums (including FLEX)
    for (position, min_needed) in min_slots
        if position == "FLEX"
            # FLEX = any RB/WR/TE
            @constraint(model,
                sum(y[i] for i in players if pos[i] in ["RB", "WR", "TE"]) >= min_needed
            )
        else
            @constraint(model,
                sum(y[i] for i in players if pos[i] == position) >= min_needed
            )
        end
    end

    # Maximums (skip FLEX max if present)
    for (position, max_allowed) in max_slots
        if position == "FLEX"
            continue
        end
        @constraint(model,
            sum(y[i] for i in players if pos[i] == position) <= max_allowed
        )
    end

    # Total roster size equals number of our picks (e.g., 16)
    @constraint(model, sum(y[i] for i in players) == length(slots))


    # --------------------------------------------------------
    # D. Bye Week Constraints (NEW FOR MODEL 2)
    # --------------------------------------------------------
    # No 2 RBs, no 2 WRs, and no 2 TEs share the same bye week.
    # Collect all bye weeks among players
    bye_weeks = unique([bye_week[i] for i in players])
    
    for b in bye_weeks
        @constraint(model,
            sum(y[i] for i in players if pos[i] == "RB" && bye_week[i] == b) <= 1
        )
        @constraint(model,
            sum(y[i] for i in players if pos[i] == "WR" && bye_week[i] == b) <= 1
        )
        @constraint(model,
            sum(y[i] for i in players if pos[i] == "TE" && bye_week[i] == b) <= 1
        )
    end

    

    # --------------------------------------------------------
    # E. Objective: Maximize Fantasy Points – Reach Penalty
    # --------------------------------------------------------
    lambda_up = 10.0      # reward upside
    lambda_bust = 20.0    # penalize bust risk

    @objective(model, Max,
    sum(
        (FPTS[i] - calculate_penalty(i, slot_to_pick[s])
         + lambda_up * upside[i]
         - lambda_bust * bust[i]) * x[i,s]
        for i in players, s in slots
    )
    )


    optimize!(model)


    # --------------------------------------------------------
    # F. Extract Optimized Team
    # --------------------------------------------------------

    opt_2_team = DataFrame(
        pick      = Int[],
        player_id = Int[],
        PLAYER    = String[],
        POSITION  = String[],
        FPTS      = Float64[],
        ADP       = Float64[],
        penalty   = Float64[],
        net_value = Float64[],
        BYE_WEEK  = Float64[]
    )

    for s in slots
        pick_num = slot_to_pick[s]

        chosen = nothing
        for i in players
            if value(x[i,s]) > 0.5
                chosen = i
                break
            end
        end

        if chosen === nothing
            # This should not happen if the model is feasible & optimal,
            # but we add a fallback for safety.
            remaining = setdiff(players, opt_2_team.player_id)
            chosen = argmax(i -> FPTS[i], remaining)
        end

        pen = calculate_penalty(chosen, pick_num)
        net_val = FPTS[chosen] - pen

        push!(opt_2_team, (
            pick_num,
            chosen,
            df.PLAYER[chosen],
            pos[chosen],
            FPTS[chosen],
            adp[chosen],
            pen,
            net_val,
            bye_week[chosen]
        ))
    end

    sort!(opt_2_team, :pick)
    return opt_2_team
end

draft_slot = 3
my_picks = generate_my_picks(draft_slot)

global_opt_team2 = solve_global_opt_model2(players, my_picks; rank_slack=0)

println("\n============ GLOBAL OPTIMIZED TEAM (Model 2, Slot $draft_slot) ============")
println(global_opt_team2[:, [:pick, :PLAYER, :POSITION, :BYE_WEEK, :ADP, :FPTS, :penalty, :net_value]])

println("\nTotal FPTS    = ", sum(global_opt_team2.FPTS))
println("Total Penalty = ", sum(global_opt_team2.penalty))
println("Net Value     = ", sum(global_opt_team2.net_value))

Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20

============ GLOBAL OPTIMIZED TEAM (Model 2, Slot 3) ============
16×8 DataFrame
 Row │ pick   PLAYER            POSITION  BYE_WEEK  ADP      FPTS     penalty  net_value
     │ Int64  String            String    Float64   Float64  Float64  Float64  Float64
─────┼───────────────────────────────────────────────────────────────────────────────────
   1 │     3  Justin Jefferson  WR             6.0      5.0    303.7  0.0        303.7
   2 │    22  Chase Brown       RB            10.0     21.2    208.6  0.0        208.6
   3 │    27  Tee Higgins       WR            10.0     31.3    247.6  0.0        247.6
   4 │    46  Chuba Hubbard     RB            14.0     46.2    187.6  0.0        187.6
   5 │    51  DeVonta Smith     WR             9.0     57.7    220.1  0.0        220.1
   6 │    70  Jaylen Waddle     WR            12.0     72.2    208.2  0.0        208.2
   7 │    75  Kaleb Johnson     RB     

In [6]:
using JuMP, Gurobi, DataFrames, StatsBase, Statistics

# --------------------------------------------------------
# STOCHASTIC OPPONENT MODELING
# --------------------------------------------------------

function generate_opponent_weights(round_num::Int)
    """
    Returns probability weights for opponent picks.
    Earlier rounds: stick closer to ADP
    Later rounds: more randomness
    """
    if round_num <= 3
        return [0.50, 0.25, 0.15, 0.07, 0.03]  # Top pick 50% likely
    elseif round_num <= 6
        return [0.40, 0.25, 0.18, 0.10, 0.07]  
    elseif round_num <= 10
        return [0.30, 0.25, 0.20, 0.15, 0.10]  
    else
        return [0.25, 0.25, 0.20, 0.15, 0.15]  # Late: almost uniform
    end
end

function simulate_one_scenario(players, my_picks, all_picks)
    """
    Simulate one complete scenario of opponent drafting.
    Returns: Dict mapping pick_number => Set of players still available
    """
    
    # Track which players are taken
    taken = Set{Int}()
    
    # For each pick in the draft
    availability = Dict{Int, Set{Int}}()
    
    for pick_num in all_picks
        # Store who's available at THIS pick (before it's made)
        availability[pick_num] = setdiff(Set(players), taken)
        
        if pick_num in my_picks
            # This is YOUR pick - don't simulate, you'll decide later
            # Just mark availability and continue
            continue
        else
            # Opponent's pick - simulate stochastically
            available_now = collect(setdiff(Set(players), taken))
            
            if isempty(available_now)
                continue
            end
            
            # Sort by ADP
            sorted_available = sort(available_now, by = i -> adp[i])
            
            # Determine round for weighting
            round_num = ceil(Int, pick_num / 12)
            weights = generate_opponent_weights(round_num)
            
            # Pick from top-K with weights
            K = min(5, length(sorted_available))
            top_k = sorted_available[1:K]
            pick_weights = weights[1:K]
            
            # Sample one player
            selected = sample(top_k, Weights(pick_weights))
            push!(taken, selected)
        end
    end
    
    return availability
end

function generate_scenarios(players, my_picks, all_picks, n_scenarios::Int)
    """
    Generate n_scenarios of opponent drafting behavior.
    Returns: Vector of availability dictionaries
    """
    
    println("\n🎲 Generating $n_scenarios draft scenarios...")
    scenarios = []
    
    for s in 1:n_scenarios
        scenario = simulate_one_scenario(players, my_picks, all_picks)
        push!(scenarios, scenario)
        
        if s % 10 == 0
            print(".")
        end
    end
    println(" Done!")
    
    return scenarios
end

# --------------------------------------------------------
# MODIFIED GLOBAL MODEL (with scenario-based availability)
# --------------------------------------------------------

function solve_global_opt_with_scenario(
        players,
        my_picks,
        availability_dict;  # NEW: availability per pick from scenario
        FPTS       = FPTS,
        pos        = pos,
        adp        = adp,
        bye_week   = bye_week,
        upside     = upside,   
        bust       = bust,
        min_slots  = min_slots,
        max_slots  = max_slots
    )

    slots = collect(1:length(my_picks))
    slot_to_pick = Dict(s => my_picks[s] for s in slots)

    model = Model(Gurobi.Optimizer)
    set_silent(model)

    # Decision variables
    @variable(model, x[players, slots], Bin)
    @variable(model, y[players], Bin)

    # --------------------------------------------------------
    # A. Slot and Uniqueness Constraints
    # --------------------------------------------------------

    for s in slots
        @constraint(model, sum(x[i,s] for i in players) == 1)
    end

    for i in players
        @constraint(model, sum(x[i,s] for s in slots) <= 1)
    end

    for i in players
        @constraint(model, y[i] >= (1/length(slots)) * sum(x[i,s] for s in slots))
        @constraint(model, y[i] <= sum(x[i,s] for s in slots))
    end

    # --------------------------------------------------------
    # A2. QB Constraint: At most 1 QB in first 5 picks
    # --------------------------------------------------------
    
    top5_slots = slots[1:min(5, length(slots))]
    @constraint(model,
        sum(x[i,s] for i in players for s in top5_slots if pos[i] == "QB") <= 1
    )

    # --------------------------------------------------------
    # B. SCENARIO-BASED AVAILABILITY (NEW!)
    # --------------------------------------------------------
    # Instead of deterministic ADP + slack, use scenario availability

    for s in slots
        pick_num = slot_to_pick[s]
        available_at_pick = availability_dict[pick_num]
        
        # If player i is NOT available in this scenario, force x[i,s] = 0
        for i in players
            if !(i in available_at_pick)
                @constraint(model, x[i,s] == 0)
            end
        end
    end

    # --------------------------------------------------------
    # C. Position Feasibility Constraints
    # --------------------------------------------------------

    for (position, min_needed) in min_slots
        if position == "FLEX"
            @constraint(model,
                sum(y[i] for i in players if pos[i] in ["RB", "WR", "TE"]) >= min_needed
            )
        else
            @constraint(model,
                sum(y[i] for i in players if pos[i] == position) >= min_needed
            )
        end
    end

    for (position, max_allowed) in max_slots
        if position == "FLEX"
            continue
        end
        @constraint(model,
            sum(y[i] for i in players if pos[i] == position) <= max_allowed
        )
    end

    @constraint(model, sum(y[i] for i in players) == length(slots))

    # --------------------------------------------------------
    # D. Bye Week Constraints
    # --------------------------------------------------------

    bye_weeks = unique([bye_week[i] for i in players])
    
    for b in bye_weeks
        @constraint(model,
            sum(y[i] for i in players if pos[i] == "RB" && bye_week[i] == b) <= 1
        )
        @constraint(model,
            sum(y[i] for i in players if pos[i] == "WR" && bye_week[i] == b) <= 1
        )
        @constraint(model,
            sum(y[i] for i in players if pos[i] == "TE" && bye_week[i] == b) <= 1
        )
    end

    # --------------------------------------------------------
    # E. Objective: Maximize Fantasy Points with penalties
    # --------------------------------------------------------

    lambda_up = 10.0
    lambda_bust = 20.0

    @objective(model, Max,
        sum(
            (FPTS[i] - calculate_penalty(i, slot_to_pick[s])
             + lambda_up * upside[i]
             - lambda_bust * bust[i]) * x[i,s]
            for i in players, s in slots
        )
    )

    optimize!(model)

    # --------------------------------------------------------
    # F. Extract Solution
    # --------------------------------------------------------

    if termination_status(model) != MOI.OPTIMAL
        return nothing, -Inf  # Infeasible scenario
    end

    team = DataFrame(
        pick      = Int[],
        player_id = Int[],
        PLAYER    = String[],
        POSITION  = String[],
        FPTS      = Float64[],
        ADP       = Float64[],
        penalty   = Float64[],
        net_value = Float64[],
        BYE_WEEK  = Float64[]
    )

    for s in slots
        pick_num = slot_to_pick[s]

        chosen = nothing
        for i in players
            if value(x[i,s]) > 0.5
                chosen = i
                break
            end
        end

        if chosen === nothing
            return nothing, -Inf
        end

        pen = calculate_penalty(chosen, pick_num)
        net_val = FPTS[chosen] - pen

        push!(team, (
            pick_num,
            chosen,
            df.PLAYER[chosen],
            pos[chosen],
            FPTS[chosen],
            adp[chosen],
            pen,
            net_val,
            bye_week[chosen]
        ))
    end

    sort!(team, :pick)
    obj_val = objective_value(model)
    
    return team, obj_val
end

# --------------------------------------------------------
# MAIN STOCHASTIC ROBUST OPTIMIZATION
# --------------------------------------------------------

function solve_stochastic_robust_model(
        players,
        my_picks,
        all_picks;
        n_scenarios = 30
    )
    
    println("\n" * "="^70)
    println("STOCHASTIC ROBUST OPTIMIZATION WITH $n_scenarios SCENARIOS")
    println("="^70)
    
    # Step 1: Generate scenarios
    scenarios = generate_scenarios(players, my_picks, all_picks, n_scenarios)
    
    # Step 2: Solve for each scenario
    println("\n🔧 Solving optimization for each scenario...")
    
    scenario_teams = []
    scenario_values = []
    scenario_fpts = []
    
    feasible_count = 0
    
    for (sc_idx, scenario) in enumerate(scenarios)
        team, obj_val = solve_global_opt_with_scenario(
            players,
            my_picks,
            scenario
        )
        
        if team !== nothing
            push!(scenario_teams, team)
            push!(scenario_values, obj_val)
            push!(scenario_fpts, sum(team.FPTS))
            feasible_count += 1
        end
        
        if sc_idx % 5 == 0
            print(".")
        end
    end
    println(" Done!")
    
    println("\n✅ Feasible scenarios: $feasible_count / $n_scenarios")
    
    if feasible_count == 0
        println("❌ No feasible solutions found!")
        return nothing
    end
    
    # Step 3: Analyze results across scenarios
    println("\n" * "="^70)
    println("SCENARIO ANALYSIS")
    println("="^70)
    
    avg_fpts = mean(scenario_fpts)
    std_fpts = std(scenario_fpts)
    min_fpts = minimum(scenario_fpts)
    max_fpts = maximum(scenario_fpts)
    
    println("Expected FPTS:    $(round(avg_fpts, digits=2))")
    println("Std Dev FPTS:     $(round(std_fpts, digits=2))")
    println("Min FPTS:         $(round(min_fpts, digits=2))")
    println("Max FPTS:         $(round(max_fpts, digits=2))")
    println("Range:            $(round(max_fpts - min_fpts, digits=2))")
    
    # Step 4: Select representative team
    # Strategy: Choose team closest to expected value
    
    expected_idx = argmin(abs.(scenario_fpts .- avg_fpts))
    representative_team = scenario_teams[expected_idx]
    
    println("\n📊 Selected representative team (closest to expected value)")
    println("Scenario #$expected_idx with FPTS = $(round(scenario_fpts[expected_idx], digits=2))")
    
    # Step 5: Player frequency analysis
    println("\n" * "="^70)
    println("PLAYER SELECTION FREQUENCY ACROSS SCENARIOS")
    println("="^70)
    
    player_frequency = Dict{Int, Int}()
    for team in scenario_teams
        for player_id in team.player_id
            player_frequency[player_id] = get(player_frequency, player_id, 0) + 1
        end
    end
    
    # Sort by frequency
    freq_sorted = sort(collect(player_frequency), by = x -> x[2], rev = true)
    
    println("\nTop players by selection frequency:")
    for (i, (player_id, count)) in enumerate(freq_sorted[1:min(20, length(freq_sorted))])
        pct = round(100 * count / feasible_count, digits=1)
        println("  $(df.PLAYER[player_id]) ($(pos[player_id])): $count/$feasible_count ($pct%)")
        
        if i >= 20
            break
        end
    end
    
    # Return results
    return Dict(
        "representative_team" => representative_team,
        "all_teams" => scenario_teams,
        "all_values" => scenario_values,
        "all_fpts" => scenario_fpts,
        "avg_fpts" => avg_fpts,
        "std_fpts" => std_fpts,
        "min_fpts" => min_fpts,
        "max_fpts" => max_fpts,
        "player_frequency" => player_frequency,
        "n_feasible" => feasible_count
    )
end

# --------------------------------------------------------
# RUN THE STOCHASTIC ROBUST MODEL
# --------------------------------------------------------

println("\n\n" * "="^70)
println("RUNNING STOCHASTIC ROBUST GLOBAL OPTIMIZATION")
println("="^70)

draft_slot = 3
my_picks = generate_my_picks(draft_slot)
all_picks = collect(1:192)  # Assuming 16 rounds × 12 teams

# Test with fewer scenarios first
n_scenarios_test = 10
println("\n🧪 Test run with $n_scenarios_test scenarios...")
test_results = solve_stochastic_robust_model(
    players, 
    my_picks, 
    all_picks; 
    n_scenarios = n_scenarios_test
)

if test_results !== nothing
    println("\n✅ Test successful! Running full optimization...")
    
    # Full run with 30 scenarios
    n_scenarios_full = 30
    final_results = solve_stochastic_robust_model(
        players, 
        my_picks, 
        all_picks; 
        n_scenarios = n_scenarios_full
    )
    
    if final_results !== nothing
        println("\n\n" * "="^70)
        println("FINAL STOCHASTIC ROBUST TEAM")
        println("="^70)
        
        final_team = final_results["representative_team"]
        println(final_team[:, [:pick, :PLAYER, :POSITION, :BYE_WEEK, :ADP, :FPTS, :penalty, :net_value]])
        
        println("\n📈 Performance Statistics:")
        println("Expected FPTS:    $(round(final_results["avg_fpts"], digits=2))")
        println("Std Dev:          $(round(final_results["std_fpts"], digits=2))")
        println("Best Case:        $(round(final_results["max_fpts"], digits=2))")
        println("Worst Case:       $(round(final_results["min_fpts"], digits=2))")
        
        # Save for comparison
        global stochastic_robust_team = final_team
        global stochastic_results = final_results
    end
else
    println("\n❌ Test failed - check model constraints")
end




RUNNING STOCHASTIC ROBUST GLOBAL OPTIMIZATION

🧪 Test run with 10 scenarios...

STOCHASTIC ROBUST OPTIMIZATION WITH 10 SCENARIOS

🎲 Generating 10 draft scenarios...
. Done!

🔧 Solving optimization for each scenario...
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
.Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-c

Dict{String, Any} with 10 entries:
  "representative_team" => 16×9 DataFrame…
  "all_values"          => Any[3136.7, 3115.8, 3143.3, 3097.23, 3115.2, 3072.7,…
  "avg_fpts"            => 2992.13
  "player_frequency"    => Dict(5=>2, 35=>11, 32=>7, 67=>1, 112=>4, 90=>28, 4=>…
  "n_feasible"          => 30
  "min_fpts"            => 2934.6
  "std_fpts"            => 22.8192
  "all_teams"           => Any[16×9 DataFrame…
  "max_fpts"            => 3025.8
  "all_fpts"            => Any[2996.7, 3025.8, 3013.3, 2977.6, 3015.2, 2992.7, …

In [8]:
###############################################################
# STARTER EVALUATION FUNCTION (WORKS WITH ALL BASELINES)
###############################################################
const STARTING_RULES = Dict(
    "QB"  => 1,
    "RB"  => 2,
    "WR"  => 2,
    "TE"  => 1,
    "K"   => 1,
    "DST" => 1,
    "FLEX" => 1
)
const FLEX_POOL = Set(["RB","WR","TE"])

"""
Evaluate starters for a given drafted roster.
Returns: (starter_dataframe, total_fpts)
"""
function evaluate_starters(team::DataFrame; proj_col::Symbol = :FPTS)
    starters = DataFrame()
    
    # =============== NON-FLEX STARTERS ===============
    for (position, count) in STARTING_RULES
        if position == "FLEX"
            continue
        end
        
        pos_pool = filter(row -> row.POSITION == position, team)
        if nrow(pos_pool) == 0
            continue
        end
        
        pos_pool_sorted = sort(pos_pool, proj_col, rev=true)
        # Take top players
        selected = pos_pool_sorted[1:min(count, nrow(pos_pool_sorted)), :]
        append!(starters, DataFrame(selected))
    end
    
    # =============== FLEX STARTER ===============
    flex_pool = filter(
        row -> (row.POSITION in FLEX_POOL) && !(row.player_id in starters.player_id),
        team
    )
    if nrow(flex_pool) > 0
        best_flex = sort(flex_pool, proj_col, rev=true)[1:1, :]
        append!(starters, DataFrame(best_flex))
    end
    
    total_fpts = sum(starters[:, proj_col])
    return starters, total_fpts
end

###############################################################
# RUN EVALUATION FOR ALL BASELINES
###############################################################
# Evaluate starters for every model
espn_starters,   espn_points   = evaluate_starters(espn_team)
myopic_starters, myopic_points = evaluate_starters(my_team)
greedy_starters, greedy_points = evaluate_starters(greedy_team)
opt_1_starters,  opt_1_points  = evaluate_starters(global_opt_team)
opt_2_starters,  opt_2_points  = evaluate_starters(global_opt_team2)
robust_starters, robust_points = evaluate_starters(stochastic_robust_team)

println("\n" * "="^60)
println("STARTER RESULTS")
println("="^60)
println("ESPN (Baseline 0) Starter Points:                ", round(espn_points, digits=1))
println("Myopic (Baseline 1) Starter Points:              ", round(myopic_points, digits=1))
println("Greedy w/ Penalty (Baseline 2) Starter Points:   ", round(greedy_points, digits=1))
println("Optimization (Model 1) Starter Points:    ", round(opt_1_points, digits=1))
println("Optimization (Model 2) Starter Points:  ", round(opt_2_points, digits=1))
println("Robust Starter Points:  ", round(robust_points, digits=1))



# ------------------------------------------------------------
# Detailed comparison table (Now includes Adaptive Model 2)
# ------------------------------------------------------------
comparison = DataFrame(
    Model = [
        "ESPN (Baseline 0)",
        "Myopic (Baseline 1)",
        "Greedy w/ Penalty (Baseline 2)",
        "Optimization (Model 1)",
        "Optimization (Model 2)",
        "Robust"
    ],
    Starter_Points = [
        espn_points,
        myopic_points,
        greedy_points,
        opt_1_points,
        opt_2_points,
        robust_points
    ],
    Total_Roster_Points = [
        sum(espn_team.FPTS),
        sum(my_team.FPTS),
        sum(greedy_team.FPTS),
        sum(global_opt_team.FPTS),
        sum(global_opt_team2.FPTS),
        sum(stochastic_robust_team.FPTS)
    ],
    Bench_Depth = [
        sum(espn_team.FPTS)        - espn_points,
        sum(my_team.FPTS)          - myopic_points,
        sum(greedy_team.FPTS)      - greedy_points,
        sum(global_opt_team.FPTS)  - opt_1_points,
        sum(global_opt_team2.FPTS)       - opt_2_points,
        sum(stochastic_robust_team.FPTS)       - robust_points
    ]
)

# Ranking by starter points
comparison.Starter_Rank = sortperm(comparison.Starter_Points, rev=true)

println("\n" * "="^60)
println("DETAILED COMPARISON")
println("="^60)
println(comparison)


# ------------------------------------------------------------
# Improvements over ESPN baseline
# ------------------------------------------------------------
espn_base = espn_points

println("\n" * "="^60)
println("IMPROVEMENTS OVER ESPN BASELINE")
println("="^60)
println("Myopic improvement:            +$(round(myopic_points - espn_base, digits=1)) points ($(round((myopic_points - espn_base)/espn_base * 100, digits=1))%)")
println("Greedy improvement:            +$(round(greedy_points  - espn_base, digits=1)) points ($(round((greedy_points  - espn_base)/espn_base * 100, digits=1))%)")
println("Optimization 1 improvement: +$(round(opt_1_points - espn_base, digits=1)) points ($(round((opt_1_points - espn_base)/espn_base * 100, digits=1))%)")
println("Optimization 2 improvement: +$(round(opt_2_points - espn_base, digits=1)) points ($(round((opt_2_points - espn_base)/espn_base * 100, digits=1))%)")
println("Robust Optimization improvement: +$(round(robust_points - espn_base, digits=1)) points ($(round((robust_points - espn_base)/espn_base * 100, digits=1))%)")


STARTER RESULTS
ESPN (Baseline 0) Starter Points:                1796.8
Myopic (Baseline 1) Starter Points:              1660.9
Greedy w/ Penalty (Baseline 2) Starter Points:   1777.9
Optimization (Model 1) Starter Points:    1794.6
Optimization (Model 2) Starter Points:  1765.4
Robust Starter Points:  1808.6

DETAILED COMPARISON
6×5 DataFrame
 Row │ Model                           Starter_Points  Total_Roster_Points  Bench_Depth  Starter_Rank
     │ String                          Float64         Float64              Float64      Int64
─────┼────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ ESPN (Baseline 0)                       1796.8               2876.3       1079.5             6
   2 │ Myopic (Baseline 1)                     1660.9               2823.3       1162.4             1
   3 │ Greedy w/ Penalty (Baseline 2)          1777.9               2883.4       1105.5             4
   4 │ Optimization (Model 1)                  1

In [9]:
using JuMP, Gurobi

"""
Evaluate team accounting for bye weeks using optimization for fill-ins.
For each week, solves an assignment problem to maximize lineup points.
"""
function evaluate_with_byes(team::DataFrame)
    
    # Step 1: Identify starters (using the simple evaluation)
    starters, _ = evaluate_starters(team)
    
    # Step 2: Identify bench (everyone not in starters)
    bench = filter(row -> !(row.player_id in starters.player_id), team)
    
    total_points_across_season = 0.0
    weekly_breakdown = DataFrame(
        Week = Int[],
        Points = Float64[],
        Byes = String[]
    )
    
    # Step 3: Iterate through all 18 weeks
    for week in 1:18
        
        # Identify who's on bye this week
        starters_on_bye = filter(row -> row.BYE_WEEK == Float64(week), starters)
        starters_active = filter(row -> row.BYE_WEEK != Float64(week), starters)
        
        # Available bench players (not on bye)
        available_bench = filter(row -> row.BYE_WEEK != Float64(week), bench)
        
        # If no one is on bye, just use starters
        if nrow(starters_on_bye) == 0
            week_points = sum(starters.FPTS) / 17.0
            total_points_across_season += week_points
            push!(weekly_breakdown, (week, week_points, "None"))
            continue
        end
        
        # Build optimization model for this week
        model = Model(Gurobi.Optimizer)
        set_silent(model)
        
        # All candidates = active starters + available bench
        all_candidates = vcat(starters_active, available_bench)
        
        # Define slots we need to fill
        slots = ["QB", "RB1", "RB2", "WR1", "WR2", "TE", "FLEX", "K", "DST"]
        
        # Decision variables: x[i, s] = 1 if player i fills slot s
        @variable(model, x[1:nrow(all_candidates), slots], Bin)
        
        # Objective: maximize total weekly points
        @objective(model, Max,
            sum(x[i, s] * (all_candidates.FPTS[i] / 17.0) 
                for i in 1:nrow(all_candidates), s in slots)
        )
        
        # Constraint 1: Each slot filled by exactly one player (or 0 if no one available)
        for s in slots
            @constraint(model, sum(x[i, s] for i in 1:nrow(all_candidates)) <= 1)
        end
        
        # Constraint 2: Each player assigned to at most one slot
        for i in 1:nrow(all_candidates)
            @constraint(model, sum(x[i, s] for s in slots) <= 1)
        end
        
        # Constraint 3: Position eligibility
        for i in 1:nrow(all_candidates)
            player_pos = all_candidates.POSITION[i]
            
            # Only QBs can fill QB slot
            if player_pos != "QB"
                @constraint(model, x[i, "QB"] == 0)
            end
            
            # Only Ks can fill K slot
            if player_pos != "K"
                @constraint(model, x[i, "K"] == 0)
            end
            
            # Only DSTs can fill DST slot
            if player_pos != "DST"
                @constraint(model, x[i, "DST"] == 0)
            end
            
            # Only TEs can fill TE slot (but can also go in FLEX)
            if player_pos != "TE"
                @constraint(model, x[i, "TE"] == 0)
            end
            
            # RBs can only fill RB slots or FLEX
            if player_pos == "RB"
                @constraint(model, x[i, "QB"] + x[i, "WR1"] + x[i, "WR2"] + x[i, "TE"] + x[i, "K"] + x[i, "DST"] == 0)
            end
            
            # WRs can only fill WR slots or FLEX
            if player_pos == "WR"
                @constraint(model, x[i, "QB"] + x[i, "RB1"] + x[i, "RB2"] + x[i, "TE"] + x[i, "K"] + x[i, "DST"] == 0)
            end
            
            # TEs can fill TE or FLEX only
            if player_pos == "TE"
                @constraint(model, x[i, "QB"] + x[i, "RB1"] + x[i, "RB2"] + x[i, "WR1"] + x[i, "WR2"] + x[i, "K"] + x[i, "DST"] == 0)
            end
        end
        
        # Solve
        optimize!(model)
        
        if termination_status(model) == MOI.OPTIMAL
            week_points = objective_value(model)
        else
            # Fallback: just count active starters
            week_points = sum(starters_active.FPTS) / 17.0
            println("Warning: No optimal solution for week $week")
        end
        
        total_points_across_season += week_points
        
        bye_players = nrow(starters_on_bye) > 0 ? join(starters_on_bye.PLAYER, ", ") : "None"
        push!(weekly_breakdown, (week, week_points, bye_players))
    end
    
    return total_points_across_season, weekly_breakdown
end

###############################################################
# Run bye week evaluation for all teams (including Opt 1 & Opt 2)
###############################################################

println("\n" * "="^70)
println("BYE WEEK ADJUSTED EVALUATION")
println("="^70)

# Evaluate teams using your bye-week simulation
espn_bye_points,   espn_weekly   = evaluate_with_byes(espn_team)
myopic_bye_points, myopic_weekly = evaluate_with_byes(my_team)
greedy_bye_points, greedy_weekly = evaluate_with_byes(greedy_team)
opt1_bye_points,   opt1_weekly   = evaluate_with_byes(global_opt_team)
opt2_bye_points,   opt2_weekly   = evaluate_with_byes(global_opt_team2)
robust_bye_points, robust_weekly = evaluate_with_byes(stochastic_robust_team)

println("\nESPN (Baseline 0) - Bye-Adjusted Points:          $(round(espn_bye_points, digits=1))")
println("Myopic (Baseline 1) - Bye-Adjusted Points:        $(round(myopic_bye_points, digits=1))")
println("Greedy (Baseline 2) - Bye-Adjusted Points:        $(round(greedy_bye_points, digits=1))")
println("Optimization (Model 1) - Bye-Adjusted:     $(round(opt1_bye_points, digits=1))")
println("Optimization (Model 2) - Bye-Adjusted:   $(round(opt2_bye_points, digits=1))")
println("Robust - Bye-Adjusted:   $(round(robust_bye_points, digits=1))")

# Build comparison table
bye_comparison = DataFrame(
    Model = [
        "ESPN",
        "Myopic",
        "Greedy w/ Penalty",
        "Optimization 1",
        "Optimization 2",
        "Robust"
    ],
    Bye_Adjusted_Points = [
        espn_bye_points,
        myopic_bye_points,
        greedy_bye_points,
        opt1_bye_points,
        opt2_bye_points,
        robust_bye_points
    ],
    Simple_Starter_Points = [
        espn_points,
        myopic_points,
        greedy_points,
        opt_1_points,
        opt_2_points,
        robust_points
    ],
    Depth_Quality = [
        espn_bye_points - espn_points,
        myopic_bye_points - myopic_points,
        greedy_bye_points - greedy_points,
        opt1_bye_points - opt_1_points,
        opt2_bye_points - opt_2_points,
        robust_bye_points - robust_points
    ]
)

# Rank by bye-adjusted points (higher = better)
bye_comparison.Rank = sortperm(bye_comparison.Bye_Adjusted_Points, rev=true)

println("\n" * "="^70)
println("FINAL COMPARISON WITH BYE WEEKS")
println("="^70)
println(bye_comparison)


BYE WEEK ADJUSTED EVALUATION
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Username
Academic license - for non-commercial